# Os orientadores são especialistas ou generalistas?

**Pergunta 3.** Olhando as bancas de defesa e qualificação do ICMC, cada orientador concentra seus alunos em poucos temas (especialista) ou orienta em várias áreas (generalista)?

**Dados.** Cada linha é uma banca (`data/eventos.csv`). O tema de cada banca (`viz/question3/temas_llm.csv`) foi escolhido pelo LLM a partir do **título**, entre os temas oficiais do programa (`viz/question3/temas_existentes.json`). Entram só os programas com árvore de temas: PPG-CCMC, PPG-Mat e PIPGES. As bancas rotuladas como "Outros" pelo LLM ficam de fora.

**Gráficos.** Os dois abaixo são independentes, cada um com seus controles:
1. **Dispersão:** um ponto por orientador, com o nº de bancas e a concentração no tema principal.
2. **Heatmap:** orientador × tema, com a fração das bancas dele em cada tema.

In [4]:
from pathlib import Path
import unicodedata
import zlib

import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML

AQUI = Path.cwd()                                  # pasta do notebook (viz/question3)
RAIZ = next(p for p in [AQUI, *AQUI.parents] if (p / "data" / "eventos.csv").exists())   # raiz do repo
ARQ_TEMAS = AQUI / "temas_llm.csv"
ESPACO = HTML('<div style="height:48px"></div>')      # espaço entre os controles e o gráfico
ROTULO = {"tema": "Tema", "macro_tema": "Macro-tema"}      # nome de cada nível nos controles, hovers e títulos
COR = {"PPG-CCMC": "#0072B2", "PIPGES": "#E69F00", "PPG-Mat": "#009E73"}   # paleta que funciona com daltonismo

## 1. Dados

Juntamos as bancas com o tema de cada uma. `tema` é o tema específico (ex: *Processamento de Linguagem Natural*) e `macro_tema` é o macro-tema que contém esse tema (ex: *Inteligência Computacional*).

In [5]:
eventos = pd.read_csv(RAIZ / "data" / "eventos.csv")[["id", "orientador"]]
temas = pd.read_csv(ARQ_TEMAS)
bancas = temas.merge(eventos, on="id")
n_outros = (bancas.tema == "Outros").sum()
bancas = bancas[bancas.tema != "Outros"]

PROGRAMAS = [p for p in COR if p in set(bancas.programa)]     # na ordem da paleta
MAX_BANCAS = int(bancas.orientador.value_counts().max())      # teto do slider "Mín. bancas": o orientador com mais bancas
print(f"{len(bancas)} bancas de {bancas.orientador.nunique()} orientadores ({n_outros} bancas 'Outros' ficaram de fora)")
bancas[["orientador", "programa", "tema", "macro_tema"]].head()

1005 bancas de 136 orientadores (41 bancas 'Outros' ficaram de fora)


,orientador,programa,tema,macro_tema
0,Renata Pontin de Mattos Fortes,PPG-CCMC,Engenharia de Software e Sistemas de Informação,Engenharia de Software e Sistemas de Informaçã...
1,Vicente Garibay Cancho,PIPGES,Modelos de Regressão,Modelos de Regressão
2,Jorge Luis Bazán Guzmán,PIPGES,Modelos de Regressão,Modelos de Regressão
3,André Carlos Ponce de Leon Ferreira de Carvalho,PPG-CCMC,Aprendizado de Máquina e Mineração de Dados,Inteligência Computacional
4,João Luis Garcia Rosa,PPG-CCMC,Sistemas Embarcados e Evolutivos,Sistemas Distribuídos e Programação Concorrent...


## 2. Como medimos

Para cada orientador, contamos quantas bancas ele tem em cada tema. Daí sai a **concentração no tema principal**: a % das bancas dele que cai no tema mais frequente dele. Perto de 100% = especialista; baixa = generalista.

O **nível** escolhe como agrupar as bancas: por **tema** ou por **macro-tema** (os dois estão definidos na seção 1). Com macro-tema, "tema principal" passa a ser o macro-tema mais frequente do orientador.

Um orientador só entra se tiver pelo menos o **mínimo de bancas** escolhido (com poucas bancas, a medida é instável).

As funções abaixo calculam isso; os gráficos vêm depois.

In [6]:
def sem_acento(s):
    return unicodedata.normalize("NFD", s).encode("ascii", "ignore").decode().lower()

def casa(nome, busca):
    return busca.strip() != "" and sem_acento(busca.strip()) in sem_acento(nome)

def curto(s, n=22):
    return s if len(s) <= n else s[: n - 1] + "…"

def perfis(programa, minimo, nivel):
    """Uma linha por orientador (perfil) e a tabela orientador x tema com o nº de bancas (conta)."""
    b = bancas if programa == "Todos" else bancas[bancas.programa == programa]
    conta = pd.crosstab(b.orientador, b[nivel])
    n = conta.sum(axis=1)
    conta, n = conta[n >= minimo], n[n >= minimo]
    p = conta.div(n, axis=0)
    perfil = pd.DataFrame({
        "n": n,
        "tema_principal": conta.idxmax(axis=1),
        "concentracao": p.max(axis=1),
    })
    perfil["programa"] = b.groupby("orientador").programa.agg(lambda s: s.value_counts().index[0]).reindex(perfil.index)
    return perfil, conta


def painel(grafico, **controles):
    """Controles + gráfico numa caixa só. Parecido com o interact, mas a cada mudança a saída é
    limpa NA HORA antes de redesenhar (o interact limpa só quando chega a próxima saída)."""
    saida = widgets.Output(layout=widgets.Layout(width="100%"))     # ocupa a largura toda da saída

    def redesenhar(*_):
        saida.clear_output()
        with saida:
            grafico(**{nome: c.value for nome, c in controles.items()})

    for c in controles.values():
        c.observe(redesenhar, names="value")
    display(widgets.VBox([*controles.values(), saida], layout=widgets.Layout(width="100%")))
    redesenhar()

## 3. Gráfico 1: dispersão

Cada ponto é um orientador (cor = programa). No eixo horizontal está o nº de bancas dele; no vertical, a **concentração no tema principal**: mais alto = mais especialista; mais baixo = mais generalista.

- Os pontos têm um pequeno deslocamento horizontal só para não se sobreporem.
- Passe o mouse num ponto para ver o orientador. Clique na legenda para esconder um programa. **Buscar** circula o orientador pelo nome.

In [7]:
def grafico_dispersao(programa="Todos", nivel="tema", minimo=4, busca=""):
    rot = ROTULO[nivel]
    perfil, _ = perfis(programa, minimo, nivel)
    perfil = perfil.assign(x=[n + ((zlib.crc32(o.encode()) % 1000) / 1000 - 0.5) * 0.36
                              for o, n in zip(perfil.index, perfil.n)])
    display(ESPACO)
    print(f"{len(perfil)} orientadores com pelo menos {minimo} bancas")

    fig = go.Figure()
    for prog in PROGRAMAS:
        q = perfil[perfil.programa == prog]
        if q.empty:
            continue
        texto = [f"<b>{o}</b> ({r.n} bancas)<br>{rot} principal: {r.tema_principal}"
                 f"<br>Concentração no {rot.lower()} principal: {r.concentracao:.0%} ({round(r.concentracao * r.n)}/{r.n})" for o, r in q.iterrows()]
        fig.add_trace(go.Scatter(x=q.x, y=q.concentracao, mode="markers", name=prog, text=texto, hoverinfo="text",
                                 marker=dict(color=COR[prog], size=10, opacity=0.85, line=dict(color="white", width=1))))

    a, b = (perfil.n.min(), perfil.n.max()) if len(perfil) else (1, MAX_BANCAS)

    achados = perfil[[casa(o, busca) for o in perfil.index]]
    if busca.strip():
        print(f"{len(achados)} encontrado(s) (circulado)" if len(achados) else "Nenhum orientador encontrado com esse nome e esse mínimo de bancas")
    if len(achados):
        fig.add_trace(go.Scatter(x=achados.x, y=achados.concentracao, mode="markers+text", text=list(achados.index),
                                 textposition="top center", showlegend=False, hoverinfo="skip",
                                 marker=dict(size=20, color="rgba(0,0,0,0)", line=dict(color="black", width=2))))

    fig.update_xaxes(title_text="Nº de bancas do orientador", title_font=dict(size=12), range=[a - 0.7, b + 0.7], dtick=1, zeroline=False)
    fig.update_yaxes(title_text=f"Concentração no {rot.lower()} principal (% das bancas)", title_font=dict(size=12), range=[0, 1.04], tickformat=".0%",
                     zeroline=False)
    fig.update_layout(height=470, margin=dict(l=70, r=20, t=10, b=120), plot_bgcolor="#fafafa",
                      legend=dict(orientation="h", y=-0.28, x=0))
    fig.show(config={"responsive": True, "displayModeBar": False})        # acompanha a largura da saída; sem a barra do plotly

In [8]:
painel(grafico_dispersao,
       programa=widgets.Dropdown(options=["Todos", *PROGRAMAS], description="Programa"),
       nivel=widgets.Dropdown(options=[(r, k) for k, r in ROTULO.items()], description="Nível"),
       minimo=widgets.IntSlider(value=min(4, MAX_BANCAS), min=1, max=MAX_BANCAS, description="Mín. bancas"),
       busca=widgets.Text(placeholder="Digite um nome", description="Buscar"))

## 4. Gráfico 2: heatmap orientador × tema

Cada coluna é um orientador e cada linha é um tema. A cor é a fração das bancas do orientador naquele tema (azul escuro = alta; **cinza = nenhuma banca**). O nº de bancas de cada um aparece ao passar o mouse.

- Um orientador especialista tem, na coluna dele, **uma célula escura** e o resto cinza; um generalista tem **várias células claras**.
- Em "Ordem", *mais especialistas primeiro* usa a concentração no tema principal do gráfico 1.
- Passe o mouse numa célula para ver os detalhes. **Buscar** contorna a coluna do orientador.

In [9]:
ESCALA = [[0, "#e5e7eb"], [0.00001, "#e5e7eb"], [0.0000101, "#c6dbef"], [0.25, "#6baed6"],
          [0.5, "#3182bd"], [0.75, "#08519c"], [1, "#08306b"]]       # zero = cinza; resto, azul claro -> escuro

def grafico_heatmap(programa="Todos", nivel="tema", minimo=4, ordem="esp", busca=""):
    rot = ROTULO[nivel]
    perfil, conta = perfis(programa, minimo, nivel)
    perfil = {
        "esp": perfil.sort_values(["concentracao", "n"], ascending=False),
        "ger": perfil.sort_values(["concentracao", "n"], ascending=[True, False]),
        "n": perfil.sort_values("n", ascending=False),
        "nome": perfil.sort_index(),
    }[ordem]
    conta = conta.loc[perfil.index]
    conta = conta.loc[:, conta.sum() > 0]                     # só as colunas (temas) que alguém usou
    pct = conta.div(perfil.n, axis=0)
    display(ESPACO)
    print(f"{len(perfil)} orientadores com pelo menos {minimo} bancas")

    z = pct.values.T                                                   # linhas = temas, colunas = orientadores
    texto = [[f"<b>{o}</b> ({perfil.n[o]} bancas)<br>{rot}: {t}<br>" + (f"Concentração: {pct.loc[o, t]:.0%} das bancas ({conta.loc[o, t]}/{perfil.n[o]})"
                                                                      if conta.loc[o, t] else "Nenhuma banca nesse tema")
              for o in conta.index] for t in conta.columns]
    fig = go.Figure(go.Heatmap(x=list(conta.index), y=list(conta.columns), z=z,
                               text=texto, hoverinfo="text", colorscale=ESCALA, zmin=0,
                               zmax=1, xgap=0, ygap=0,
                               colorbar=dict(title="% das bancas",
                                             tickformat=".0%",
                                             x=1.01, xanchor="left", y=1, yanchor="top",
                                             lenmode="pixels", len=260, thickness=14)))
    idx = [i for i, o in enumerate(conta.index) if casa(o, busca)]
    if busca.strip():
        print(f"{len(idx)} encontrado(s) (coluna com contorno preto)" if idx else "Nenhum orientador encontrado com esse nome e esse mínimo de bancas")
    for i in idx:
        fig.add_shape(type="rect", xref="x", yref="y", x0=i - 0.5, x1=i + 0.5, y0=-0.5, y1=len(conta.columns) - 0.5,
                      line=dict(color="black", width=2.5), layer="above")
    topo = int(0.82 * 5.5 * max((len(o) for o in conta.index), default=0)) + 40      # altura do maior nome girado em 55 graus
    fig.update_xaxes(tickangle=-55, side="top", automargin=True, tickfont=dict(size=9), type="category", fixedrange=True)
    fig.update_yaxes(autorange="reversed", automargin=True, tickfont=dict(size=9), fixedrange=True, tickvals=list(conta.columns),
                     ticktext=[curto(t) for t in conta.columns])
    fig.update_layout(height=max(380, topo + 26 * len(conta.columns) + 40), margin=dict(l=130, r=20, t=topo, b=40), dragmode=False,
                      plot_bgcolor="white")
    fig.show(config={"responsive": True, "displayModeBar": False})        # acompanha a largura da saída

In [10]:
painel(grafico_heatmap,
       programa=widgets.Dropdown(options=["Todos", *PROGRAMAS], description="Programa"),
       nivel=widgets.Dropdown(options=[(r, k) for k, r in ROTULO.items()], description="Nível"),
       minimo=widgets.IntSlider(value=min(4, MAX_BANCAS), min=1, max=MAX_BANCAS, description="Mín. bancas"),
       ordem=widgets.Dropdown(options=[("Mais especialistas primeiro", "esp"), ("Mais generalistas primeiro", "ger"),
                                       ("Nº de bancas", "n"), ("Nome", "nome")], description="Ordem"),
       busca=widgets.Text(placeholder="Digite um nome", description="Buscar"))

## 5. Limitações

- **Temas rotulados por LLM, a partir do título.** Títulos curtos ou genéricos podem cair no tema errado
- **Um tema por banca.** Uma banca que mistura dois assuntos conta só para o principal.
- **Poucas bancas por orientador.** A mediana é de 3. Com poucas bancas, "100% em um tema" pode ser coincidência; por isso existe o mínimo de bancas.
- **Só 3 programas.** MECAI e PROFMAT não têm árvore de temas e ficaram de fora, assim como as bancas "Outros".